# MoE-MedIR — Google Colab Training Notebook

> **Paper**: Bridging the Intra-Modal Heterogeneity Gap: Mixture-of-Experts for Multi-Modality Medical Image Retrieval  
> **Conference**: ICARCV 2026

**Setup**: Runtime → Change runtime type → T4 GPU  
Run cells top-to-bottom.

## Cell 1 — Check GPU

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
else:
    raise RuntimeError('No GPU — Runtime → Change runtime type → T4 GPU')

## Cell 2 — Install Packages

In [ ]:
%%capture
!pip install medmnist open_clip_torch timm pytorch-metric-learning scikit-learn matplotlib seaborn
print('Done.')

## Cell 3 — Clone Project từ GitHub

**Bước 1**: Tạo repo trên GitHub rồi đổi `REPO_URL` bên dưới.  
**Bước 2**: Mỗi lần mở lại Colab, cell này sẽ `git pull` để lấy code mới nhất.

In [ ]:
import os

REPO_URL     = 'https://github.com/YOUR_USERNAME/moe_medir.git'  # <-- đổi
PROJECT_ROOT = '/content/moe_medir'

if os.path.exists(PROJECT_ROOT):
    !git -C {PROJECT_ROOT} pull
else:
    !git clone {REPO_URL} {PROJECT_ROOT}

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
!git log --oneline -3

## Cell 4 — Mount Google Drive (lưu results)

Optional — backup checkpoint và results về Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS_BACKUP = '/content/drive/MyDrive/moe_medir_results'
import os; os.makedirs(RESULTS_BACKUP, exist_ok=True)
print('Drive mounted. Backup path:', RESULTS_BACKUP)

## Cell 5 — Sanity Check

Kiểm tra imports + config + model **trước khi download data**.

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, 'test_imports.py'], capture_output=False)
if result.returncode != 0:
    print('\n[!] Fix lỗi trên trước khi tiếp tục!')
else:
    print('\nAll checks passed — safe to proceed.')

## Cell 6 — Extract Features

Download MedMNIST 28px (~1.2GB) + extract 1024-dim features. Chạy 1 lần (~8 phút).

In [ ]:
import os
if os.path.exists('data/features/pathmnist_train_feat.npy'):
    print('Features already extracted — skipping.')
else:
    !python extract_features.py

## Cell 7 — Zero-Shot Baselines

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'baselines/zeroshot.py'], check=True)

## Cell 8 — HashNet Baseline

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'baselines/hashnet.py'], check=True)

## Cell 9 — Linear Baseline

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'train.py', '--model', 'linear', '--epochs', '50'], check=True)
subprocess.run([sys.executable, 'eval/evaluate.py', '--model', 'linear'], check=True)

## Cell 10 — MLP Baseline

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'train.py', '--model', 'mlp', '--epochs', '50'], check=True)
subprocess.run([sys.executable, 'eval/evaluate.py', '--model', 'mlp'], check=True)

## Cell 11 — MoE-MedIR (Main Model)

~15 phút trên T4. Log `avg mAP@R` mỗi 5 epochs.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'train.py', '--model', 'moe', '--epochs', '50'], check=True)

## Cell 12 — Full Test Evaluation

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'eval/evaluate.py', '--model', 'moe'], check=True)

## Cell 13 — Compile Comparison Table

In [ ]:
import subprocess, sys, pandas as pd
subprocess.run([sys.executable, 'baselines/compile_table.py'], check=True)
df = pd.read_csv('results/table_main.csv')
print(df.to_string(index=False))

## Cell 14 — Print LaTeX Table

In [ ]:
with open('results/table_main.tex') as f:
    print(f.read())

## Cell 15 — Expert Routing Heatmap

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'analysis/routing_heatmap.py'], check=True)

## Cell 16 — t-SNE Visualisation

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'analysis/tsne_plot.py', '--model', 'moe', '--max_samples', '400'], check=True)

## Cell 17 — Copy Results to Google Drive

In [ ]:
import shutil
shutil.copytree('results', RESULTS_BACKUP, dirs_exist_ok=True)
print('Results copied to', RESULTS_BACKUP)